# **Phase 3 — KPI Calculations**

In [1]:
import warnings
warnings.filterwarnings("ignore")

import time
import os
from pathlib import Path # for system file paths related tasks
import pandas as pd
import numpy as np

In [2]:
pd.set_option("display.float_format", "{:.2f}".format)
np.set_printoptions(precision = 2, suppress = True)

In [3]:
ROOT_DIR = Path.cwd().parent
os.chdir(ROOT_DIR)

from src.config import Database,Reports
os.chdir(ROOT_DIR/Database)
from database_connector import engine

C:\Users\KISHORE\OneDrive\Pictures\Documents\internmo\StreamFlix-Content-Analytics


2026-09-21 16:47:14,101 - INFO - loading ENV variables...
2026-09-21 16:47:14,103 - INFO - URL CREATED!!
2026-09-21 16:47:14,105 - INFO - Creating DB Engine...
2026-09-21 16:47:14,242 - INFO - Engine Created!!


In [4]:
watch_history_data = pd.read_sql_query(
    """
    SELECT * 
    FROM watch_history;
    
    """,
    con = engine
)
watch_history_data.head()

,watch_id,subscriber_id,title_id,watch_date,device,region,content_duration_min,watch_duration_min,completion_pct,completed,watch_duration_hourse
0,2657,SUB109603,TTL205011,2025-09-28,Laptop,Europe,1107,1083.80,97.90,True,18.06
1,4763,SUB106721,TTL203006,2025-03-06,Mobile,Europe,1197,685.80,57.30,False,11.43
2,4833,SUB106476,TTL202356,2025-02-23,Mobile,Europe,195,24.00,12.30,False,0.40
3,8529,SUB109887,TTL206948,2026-04-12,Smart TV,Europe,1092,1028.20,94.20,True,17.14
4,9909,SUB106046,TTL207307,2025-07-05,Tablet,Europe,117,110.40,94.30,True,1.84


## **1) KPI: Total Watch Hours (TWH)**
### **KPI Purpose:** Track Subscribers Total engagement with platform content.
### **KPI Definition:** Total Watch Hours = SUM(watch_duration_min) ÷ 60
### **Data Sources:**
- watch_history Table
### **Exspected Output:** a single kpi value(Float) represeting Total Watch Hours of platform at particular time frame.

In [5]:
Total_Watch_hours = (watch_history_data["watch_duration_min"] / 60.0).sum()
print("TWH :", round(Total_Watch_hours,2))

TWH : 3333467.73


## **2) KPI: Active Rate**
### **KPI Purpose:** Track Health of the subscriber base in platform.
### **KPI Definition:** Active Rate = Active Subscribers ÷ Total × 100
### **Data Sources:**
- subscribers Table
### **Exspected Output:** a single kpi value(Float)  represeting Active Rate in platform at particular time frame.

In [6]:
subscribers_data = pd.read_sql_query(
    """
    SELECT * 
    FROM subscribers;
    
    """,
    con = engine
)
subscribers_data.head()

,subscriber_id,signup_date,country,region,age,gender,plan_type,monthly_price_usd,household_size,primary_device,payment_method,tenure_months,is_active,churn_date
0,SUB100000,2023-01-09,France,Europe,22,Female,Basic with Ads,6.99,1,Mobile,Credit Card,41,True,None
1,SUB100001,2020-02-05,United Kingdom,Europe,48,Male,Basic with Ads,6.99,1,Smart TV,Credit Card,76,False,2022-12-08
2,SUB100002,2022-04-15,Italy,Europe,35,Male,Standard,15.49,2,Streaming Stick,Gift Card,50,True,None
3,SUB100003,2024-10-10,Spain,Europe,49,Female,Basic with Ads,6.99,1,Laptop,Credit Card,19,True,None
4,SUB100004,2020-04-12,United States,North America,36,Female,Basic with Ads,6.99,1,Mobile,Credit Card,74,False,2021-12-15


In [7]:
active_subscribers_data = subscribers_data[subscribers_data["is_active"] == True]
Active_Rate = (len(active_subscribers_data) / len(subscribers_data))*100
print(f"Active Rate : {round(Active_Rate,2)}%")

Active Rate : 74.66%


## **3) KPI: Churn Rate**
### **KPI Purpose:** Track & Check churn percentage in platform, in order to take appropriate actions when ever needed(Retentions).
### **KPI Definition:** Churn Rate = Inactive Subscribers ÷ Total × 100
### **Data Sources:**
- subscribers Table
### **Exspected Output:** a single kpi value(Float)  represeting Churn Rate in platform at particular time frame.

In [8]:
inactive_subscribers_data = subscribers_data[subscribers_data["is_active"] == False]
Churn_Rate = (len(inactive_subscribers_data) / len(subscribers_data))*100
print(f"Active Rate : {round(Churn_Rate,2)}%")

Active Rate : 25.34%


## **1) KPI: Avg Completion Rate**
### **KPI Purpose:** Track & answer this type of question Are people finishingcontent?
### **KPI Definition:** Avg Completion Rate = AVG(completion_pct)
### **Data Sources:**
- subscribers Table
### **Exspected Output:** a single kpi value(Float)  represeting Avg Completion Rate in platform at particular time frame.

In [9]:
average_completion_rate = watch_history_data["completion_pct"].mean()
print(f"Avg Completion Rate : {round(average_completion_rate,2)}%")

Avg Completion Rate : 65.31%


## **4) KPI: Monthly Recurring Revenue (MRR)**
### **KPI Purpose:** shows Actual recurring money earned.
### **KPI Definition:** Monthly Recurring Revenue  = SUM(monthly_price) for active subs
### **Data Sources:**
- subscribers Table
### **Exspected Output:** a single kpi value(Float)  represeting Monthly Recurring Revenue in platform at particular time frame.

In [10]:
monthly_recurring_revenue = active_subscribers_data.monthly_price_usd.sum()
print(f"Monthly Recurring Revenue : {round(monthly_recurring_revenue,2)}%")

Monthly Recurring Revenue : 175868.51%


## **5) KPI: Average revenue per active user (ARPU)**
### **KPI Purpose:** shows Average revenue per active user from platform.
### **KPI Definition:** Average revenue per active user  = MRR ÷ Active Subscribers
### **Data Sources:**
- subscribers Table
### **Exspected Output:** a single kpi value(Float) represeting Average revenue per active user in platform at particular time frame.

In [11]:
arpu = monthly_recurring_revenue / len(active_subscribers_data)
print(f"Monthly Recurring Revenue : {round(arpu,2)}%")

Monthly Recurring Revenue : 15.7%


## **6) KPI: Avg Watch Time / Subscriber**
### **KPI Purpose:** shows How engaged the average subscriber is.
### **KPI Definition:** Avg Watch Time / Subscriber  = Total Watch Hours ÷ Active Subs
### **Data Sources:**
- subscribers Table
### **Exspected Output:** a single kpi value(Float) represeting Avg Watch Time / Subscriber in platform at particular time frame.

In [12]:
Avg_Watch_Time = Total_Watch_hours / len(active_subscribers_data)
print(f"Avg Watch Time/Subscriber : {round(Avg_Watch_Time,2)}")

Avg Watch Time/Subscriber : 297.66
